# UQ for LLMs — Benchmarking: Which Uncertainty Method Best Predicts Model Errors?

**Course:** Big Data & Text Mining — University of Bologna, A.Y. 2025/26  
**Dataset:** MedQA-USMLE + MMLU-Medical  
**Model:** MedGemma / configurable  
**Libraries:** lm-polygraph · uqlm  
**Primary metric:** PRR (Prediction-Rejection Ratio) — as recommended in the TACL paper and AAAI-2026 tutorial

---

## Motivation

The tutorial section (black-box and white-box notebooks) shows *how* to use each UQ technique on individual prompts.  
This notebook answers a fundamentally different question:

> **"How well does each UQ technique predict when the model will make a clinical error?"**

We are *not* evaluating the LLM's accuracy. We are evaluating how *informative* the uncertainty scores are — do high-uncertainty samples correspond to wrong answers more often than low-uncertainty ones?

## Evaluation Pipeline (following the TACL paper)

```
1. Load medical MCQ test set (MedQA / MMLU-Medical)
2. For each question:
   a. Generate model answer
   b. Compute uncertainty score with each UQ technique
   c. Check if answer is correct (vs ground truth)
3. For each UQ technique:
   a. Sort samples by uncertainty score
   b. Progressively reject the most uncertain predictions
   c. Measure how accuracy improves → PRR curve
4. Rank all techniques by PRR → comparison table
```

## Metrics

| Metric | What it measures |
|---|---|
| **PRR** (Prediction-Rejection Ratio) | How much accuracy improves when uncertain predictions are filtered out (primary) |
| **AUROC** | How well uncertainty score distinguishes correct from incorrect answers |
| **AUARC** | Area under the accuracy-rejection curve (unnormalized PRR) |
| **Acc@50%** | Accuracy when rejecting the 50% most uncertain predictions |

## 1. Environment Setup

In [ ]:
# Install / upgrade required packages
!pip install -q lm-polygraph uqlm datasets scikit-learn tqdm matplotlib seaborn scipy pandas nest_asyncio


In [ ]:
import os, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path
from scipy import stats
from sklearn.metrics import roc_auc_score
import asyncio
import nest_asyncio
nest_asyncio.apply()   # allow nested event loops in Jupyter / Colab
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

print("All imports OK.")


## 2. Configuration

All key parameters are in a single cell — change them here to adapt the benchmark to a different model or dataset size.

> **Quick start:** Set `N_SAMPLES = 50` for a fast smoke-test (~ 10 min on GPU).  
> For paper-faithful results use `N_SAMPLES = 200+`.

In [ ]:
# ─── MODEL ────────────────────────────────────────────────────────────────
MODEL_ID = "google/medgemma-4b-it"   # change to any HuggingFace causal LM
POLYGRAPH_MODE = "black"             # "black" = no logits needed  |  "white" = full access
                              # NOTE: white-box techniques require "white" mode

# ─── DATASETS ─────────────────────────────────────────────────────────────
N_SAMPLES     = 100    # samples per dataset (increase for paper-faithful results)
DATASETS = {
    "MedQA":        {"hf_path": "GBaker/MedQA-USMLE-4-options",   "split": "test"},
    "MMLU-Medical": {"hf_path": "cais/mmlu", "hf_name": "clinical_knowledge", "split": "test"},
}

# ─── UQ TECHNIQUES TO BENCHMARK ───────────────────────────────────────────────
# Format: (library, technique_name, granularity, display_label, family)
# Families: Verbalized | Consistency-BB | Info-WB | Introspective-WB
BENCHMARK_TECHNIQUES = [
    # Black-box — Verbalized
    ("polygraph", "linguistic_1s",           "sequence", "Linguistic-1S",    "Verbalized"),
    # Black-box — Consistency (lm-polygraph) — graph-based (AAAI-2026 recommended)
    ("polygraph", "eig_val_laplacian",       "sequence", "EigLaplacian",     "Consistency-BB"),
    ("polygraph", "degmat",                  "sequence", "DegMat",           "Consistency-BB"),
    ("polygraph", "eccentricity",            "sequence", "Eccentricity",     "Consistency-BB"),
    # Black-box — Consistency (lm-polygraph) — graph-free
    ("polygraph", "semantic_entropy",        "sequence", "SemanticEntropy",  "Consistency-BB"),
    ("polygraph", "lexical_similarity",      "sequence", "LexicalSimilarity","Consistency-BB"),
    ("polygraph", "sar",                     "sequence", "SAR",              "Consistency-BB"),
    ("polygraph", "semantic_density",        "sequence", "SemanticDensity",  "Consistency-BB"),
    # Black-box — Consistency (UQLM)
    ("uqlm",      "entailment",              "sequence", "UQLM-Entailment",  "Consistency-BB"),
    ("uqlm",      "cosine_sim",              "sequence", "UQLM-CosineSim",   "Consistency-BB"),
    # White-box — Information-theoretic  (requires POLYGRAPH_MODE = "white")
    ("polygraph", "msp",                     "sequence", "MSP",              "Info-WB"),
    ("polygraph", "mean_token_entropy",      "sequence", "MeanTokenEntropy", "Info-WB"),
    # White-box — Introspective           (requires POLYGRAPH_MODE = "white")
    ("polygraph", "mahalanobis_distance_seq","sequence", "Mahalanobis",      "Introspective-WB"),
]

# ─── SAMPLING SETTINGS ────────────────────────────────────────────────────────
NUM_SAMPLES_UQ = 5      # K samples for consistency methods
TEMPERATURE    = 0.7    # sampling temperature

# ─── OUTPUT ───────────────────────────────────────────────────────────────────
RESULTS_DIR  = Path("benchmark_results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Model: {MODEL_ID}")
print(f"Mode:  {POLYGRAPH_MODE}")
print(f"Samples per dataset: {N_SAMPLES}")
print(f"Techniques to benchmark: {len(BENCHMARK_TECHNIQUES)}")
for lib, tech, gran, label, family in BENCHMARK_TECHNIQUES:
    print(f"  [{family:<18s}] {label}")


## 3. Dataset Loading

We benchmark on two complementary medical MCQ datasets:

| Dataset | Questions | Options | Source |
|---|---|---|---|
| **MedQA-USMLE** | 1,273 test | 4 | US Medical Licensing Exam |
| **MMLU-Medical** | 265 test | 4 | Massive Multitask Language Understanding (clinical knowledge subset) |

Both datasets have clear ground-truth answer keys — perfect for computing correctness labels.  
We sample `N_SAMPLES` from each test split for efficiency.

In [ ]:
from datasets import load_dataset

def load_medqa(n: int, seed: int = SEED):
    """Load MedQA-USMLE 4-option MCQ test set."""
    ds = load_dataset("GBaker/MedQA-USMLE-4-options", split="test")
    ds = ds.shuffle(seed=seed).select(range(min(n, len(ds))))
    records = []
    for row in ds:
        opts = row["options"]  # dict: {"A": ..., "B": ..., "C": ..., "D": ...}
        records.append({
            "dataset": "MedQA",
            "question": row["question"],
            "options":  opts,
            "answer_key": row["answer_idx"],   # "A" / "B" / "C" / "D"
        })
    return records


def load_mmlu_medical(n: int, seed: int = SEED):
    """Load MMLU clinical_knowledge subset."""
    ds = load_dataset("cais/mmlu", "clinical_knowledge", split="test")
    ds = ds.shuffle(seed=seed).select(range(min(n, len(ds))))
    LETTERS = ["A", "B", "C", "D"]
    records = []
    for row in ds:
        opts = {LETTERS[i]: row["choices"][i] for i in range(len(row["choices"]))}
        records.append({
            "dataset": "MMLU-Medical",
            "question": row["question"],
            "options":  opts,
            "answer_key": LETTERS[row["answer"]],
        })
    return records


def format_mcq_prompt(record: dict) -> str:
    """Format a MCQ record into a clinical prompt for the model."""
    opts_str = "\n".join(f"{k}. {v}" for k, v in record["options"].items())
    return (
        "You are a clinical expert. Answer the following medical multiple-choice question "
        "by selecting the single best option. Output only the letter (A, B, C, or D).\n\n"
        f"Question: {record['question']}\n\n{opts_str}\n\nAnswer:"
    )


# ── Load datasets ──────────────────────────────────────────────────────────
print(f"Loading {N_SAMPLES} samples from MedQA ...")
medqa_records = load_medqa(N_SAMPLES)

print(f"Loading {N_SAMPLES} samples from MMLU-Medical ...")
mmlu_records  = load_mmlu_medical(N_SAMPLES)

all_records = medqa_records + mmlu_records
print(f"\nTotal samples: {len(all_records)}")

# Show a sample
ex = medqa_records[0]
print("\n── Example (MedQA) ─────────────────────────────────────────────")
print(format_mcq_prompt(ex))
print(f"Ground truth: {ex['answer_key']}")

## 4. Model & UQ Engine Setup

In [ ]:
# ── lm-polygraph model wrapper ────────────────────────────────────────────
from lm_polygraph.utils.model import WhiteboxModel
from lm_polygraph.utils.generation_parameters import GenerationParameters
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print(f"Loading model: {MODEL_ID}")
print("Using 4-bit quantization (bitsandbytes) to fit on a single GPU...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

polygraph_model = WhiteboxModel.from_pretrained(
    MODEL_ID,
    generation_params=GenerationParameters(
        temperature=TEMPERATURE,
        do_sample=True,
        num_return_sequences=NUM_SAMPLES_UQ,
        max_new_tokens=32,      # short for MCQ — just need the letter + brief explanation
    ),
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager",   # required for some polygraph stat calculators
)

print(f"Model loaded on: {polygraph_model.device}")

In [ ]:
# ── UQLM LangChain wrapper ────────────────────────────────────────────────
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline as hf_pipeline

hf_pipe = hf_pipeline(
    "text-generation",
    model=MODEL_ID,
    model_kwargs={"torch_dtype": torch.bfloat16, "load_in_4bit": True},
    max_new_tokens=32,
    device_map="auto",
    do_sample=True,
    temperature=TEMPERATURE,
)
langchain_llm = HuggingFacePipeline(pipeline=hf_pipe)
print("UQLM LangChain wrapper ready.")

In [ ]:
# Shared registry — populated by the next cell
UQ_REGISTRY = {}

print("UQ_REGISTRY ready.")


In [ ]:
# ── Register all benchmark techniques ─────────────────────────────────────────
from lm_polygraph.estimators import (
    Linguistic1S,
    EigValLaplacian, DegMat, Eccentricity,
    SemanticEntropy, LexicalSimilarity,
    SAR, SemanticDensity,
    MaximumSequenceProbability, MeanTokenEntropy,
    MahalanobisDistanceSeq,
)
from uqlm import BlackBoxUQ

ESTIMATOR_MAP = {
    "linguistic_1s":            Linguistic1S,
    "eig_val_laplacian":        EigValLaplacian,
    "degmat":                   DegMat,
    "eccentricity":             Eccentricity,
    "semantic_entropy":         SemanticEntropy,
    "lexical_similarity":       LexicalSimilarity,
    "sar":                      SAR,
    "semantic_density":         SemanticDensity,
    "msp":                      MaximumSequenceProbability,
    "mean_token_entropy":       MeanTokenEntropy,
    "mahalanobis_distance_seq": MahalanobisDistanceSeq,
}

DEFAULT_KWARGS = {
    "eig_val_laplacian":  {"similarity_score": "NLI_score", "affinity": "entail"},
    "degmat":             {"similarity_score": "NLI_score", "affinity": "entail"},
    "eccentricity":       {"similarity_score": "NLI_score", "affinity": "entail"},
    "lexical_similarity": {"metric": "rougeL"},
    "semantic_entropy":   {"class_probability_estimation": "frequency"},
    "sar":                {"similarity_score": "NLI_score"},
}

for lib, tech, gran, label, family in BENCHMARK_TECHNIQUES:
    key = f"{lib}_{tech}"
    if lib == "polygraph" and tech in ESTIMATOR_MAP:
        mode = "white" if family.endswith("WB") else "black"
        UQ_REGISTRY[key] = {
            "library": "lm_polygraph",
            "supported_granularity": [gran],
            "min_required_mode": mode,
            "estimator_class": ESTIMATOR_MAP[tech],
            "default_kwargs": DEFAULT_KWARGS.get(tech, {}),
            "display_label": label,
            "family": family,
        }
    elif lib == "uqlm":
        UQ_REGISTRY[key] = {
            "library": "uqlm",
            "supported_granularity": [gran],
            "min_required_mode": "black",
            "wrapper_class": BlackBoxUQ,
            "display_label": label,
            "family": family,
        }

print(f"Registered {len(UQ_REGISTRY)} techniques for benchmarking.")
for k, v in UQ_REGISTRY.items():
    print(f"  {k:<45s} [{v['family']}]")


## 5. Batch Inference & UQ Scoring

For each sample we:
1. Run model generation (greedy decode → extract answer letter)
2. Run each UQ technique → record uncertainty score
3. Compare extracted letter to ground truth → correctness label
4. Save results progressively to CSV (checkpoint in case of runtime crash)

> **Runtime estimate:** ~30 s/sample × 200 samples × (# UQ techniques that need K samplings) on a T4 GPU.  
> Reduce `N_SAMPLES` or `NUM_SAMPLES_UQ` if time is limited.

In [ ]:
import re, torch
from lm_polygraph.estimators import estimate_uncertainty

# ── Answer extraction ──────────────────────────────────────────────────────────
def extract_answer_letter(generated: str):
    """Extract A/B/C/D from model output (greedy generation)."""
    text = generated.strip()
    if text and text[0].upper() in "ABCD":
        return text[0].upper()
    m = re.search(r"\b([A-D])\b", text, re.IGNORECASE)
    return m.group(1).upper() if m else None

# ── Greedy generation ─────────────────────────────────────────────────────────
def greedy_generate(polygraph_model, prompt: str, max_new_tokens: int = 32) -> str:
    """
    Generate a single greedy response via the underlying HuggingFace model.
    Bypasses lm-polygraph's wrapper so generation_parameters (do_sample,
    num_return_sequences) do not interfere with correctness evaluation.
    """
    tok    = polygraph_model.tokenizer
    mdl    = polygraph_model.model
    inputs = tok(prompt, return_tensors="pt").to(mdl.device)
    n_in   = inputs["input_ids"].shape[1]
    with torch.no_grad():
        ids = mdl.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(ids[0][n_in:], skip_special_tokens=True)

# ── lm-polygraph UQ scorer ─────────────────────────────────────────────────────
def score_polygraph(prompt: str, tech_name: str, tech_info: dict) -> float:
    """Run one lm-polygraph estimator on a single prompt. Returns uncertainty score."""
    kwargs    = dict(tech_info.get("default_kwargs", {}))
    estimator = tech_info["estimator_class"](**kwargs)
    try:
        output = estimate_uncertainty(polygraph_model, estimator, inp=prompt)
        # estimate_uncertainty may return a scalar, dict, or list — handle all cases
        if isinstance(output, dict):
            score = float(next(iter(output.values())))
        elif hasattr(output, "__len__"):
            score = float(output[0])
        else:
            score = float(output)
        return score
    except Exception as e:
        print(f"    [warn] {tech_name}: {e}")
        return float("nan")

# ── UQLM async scorer ──────────────────────────────────────────────────────────
async def _score_uqlm_async(prompt: str, tech_name: str, tech_info: dict) -> float:
    """Run one UQLM scorer asynchronously. Returns uncertainty = 1 − confidence."""
    try:
        scorer_name = tech_name.replace("uqlm_", "")
        wrapper = tech_info["wrapper_class"](
            llm=langchain_llm,
            scorers=[scorer_name],
        )
        result     = await wrapper.generate_and_score(
            prompts=[prompt], num_responses=NUM_SAMPLES_UQ
        )
        res_dict   = result.to_dict()
        confidence = res_dict["data"][scorer_name][0]
        return 1.0 - float(confidence)
    except Exception as e:
        print(f"    [warn] {tech_name}: {e}")
        return float("nan")

def score_uqlm(prompt: str, tech_name: str, tech_info: dict) -> float:
    """Synchronous wrapper for _score_uqlm_async (works in Jupyter via nest_asyncio)."""
    return asyncio.get_event_loop().run_until_complete(
        _score_uqlm_async(prompt, tech_name, tech_info)
    )

print("Scoring functions defined.")
print("  greedy_generate()    → greedy text via HuggingFace model.generate()")
print("  score_polygraph()    → lm-polygraph UQ score")
print("  score_uqlm()         → UQLM UQ score (async, via nest_asyncio)")


In [ ]:
# ── Main benchmark loop ───────────────────────────────────────────────────────
# Results are appended row-by-row and checkpointed every 10 samples so partial
# results survive runtime disconnects (e.g., Colab timeout).

CHECKPOINT_PATH = RESULTS_DIR / "benchmark_raw.csv"

# Load existing checkpoint if available (resume from interruption)
if CHECKPOINT_PATH.exists():
    existing_df = pd.read_csv(CHECKPOINT_PATH)
    done_ids    = set(zip(existing_df["dataset"], existing_df["sample_id"]))
    all_rows    = existing_df.to_dict("records")
    print(f"Resuming from checkpoint: {len(all_rows)} rows already done.")
else:
    done_ids = set()
    all_rows = []

# ── White-box technique keys (skip if running in black-box mode) ──────────────
WB_TECHNIQUES = {lib + "_" + tech for lib, tech, _, _, fam in BENCHMARK_TECHNIQUES
                 if fam.endswith("WB")}

# ── Main loop ─────────────────────────────────────────────────────────────────
for dataset_name, records in [("MedQA", medqa_records), ("MMLU-Medical", mmlu_records)]:
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_name}  ({len(records)} samples)")
    print("="*60)

    for sample_id, record in enumerate(tqdm(records, desc=dataset_name)):
        uid = (dataset_name, sample_id)
        if uid in done_ids:
            continue

        prompt = format_mcq_prompt(record)
        row = {
            "dataset":    dataset_name,
            "sample_id":  sample_id,
            "answer_key": record["answer_key"],
        }

        # ── Step 1: greedy generation for correctness label ───────────────────
        # Uses the raw HuggingFace model directly — generation_parameters on the
        # WhiteboxModel wrapper do NOT affect this call.
        try:
            greedy_text = greedy_generate(polygraph_model, prompt, max_new_tokens=32)
        except Exception as e:
            print(f"  [warn] greedy_generate failed: {e}")
            greedy_text = ""

        extracted_letter    = extract_answer_letter(greedy_text)
        row["generated_text"]   = greedy_text[:120]
        row["predicted_letter"] = extracted_letter
        row["correct"]          = (
            int(extracted_letter == record["answer_key"]) if extracted_letter else 0
        )

        # ── Step 2: UQ scoring — restore sampling parameters for K-sample methods
        polygraph_model.generation_parameters.do_sample          = True
        polygraph_model.generation_parameters.temperature        = TEMPERATURE
        polygraph_model.generation_parameters.num_return_sequences = NUM_SAMPLES_UQ

        for lib, tech, gran, label, family in BENCHMARK_TECHNIQUES:
            reg_key   = f"{lib}_{tech}"
            tech_info = UQ_REGISTRY.get(reg_key)
            if tech_info is None:
                row[label] = float("nan")
                continue

            # Skip white-box techniques in black-box mode
            if reg_key in WB_TECHNIQUES and POLYGRAPH_MODE == "black":
                row[label] = float("nan")
                continue

            if lib == "polygraph":
                row[label] = score_polygraph(prompt, reg_key, tech_info)
            else:
                row[label] = score_uqlm(prompt, reg_key, tech_info)

        all_rows.append(row)

        # ── Checkpoint every 10 samples ───────────────────────────────────────
        if len(all_rows) % 10 == 0:
            pd.DataFrame(all_rows).to_csv(CHECKPOINT_PATH, index=False)

# Final save
results_df = pd.DataFrame(all_rows)
results_df.to_csv(CHECKPOINT_PATH, index=False)
print(f"\nDone. {len(results_df)} rows saved to {CHECKPOINT_PATH}")
print(results_df[["dataset","sample_id","answer_key","predicted_letter","correct"]].head(10))


## 6. Correctness Analysis

In [ ]:
# ── Load results (from checkpoint) ───────────────────────────────────────
results_df = pd.read_csv(RESULTS_DIR / "benchmark_raw.csv")

TECH_LABELS = [label for _, _, _, label, _ in BENCHMARK_TECHNIQUES]
TECH_FAMILY = {label: fam for _, _, _, label, fam in BENCHMARK_TECHNIQUES}

# ── Overall accuracy per dataset ──────────────────────────────────────────
print("=" * 55)
print(f"{'Dataset':<20} {'N':>5} {'Accuracy':>10} {'Parse Rate':>12}")
print("-" * 55)
for ds in results_df["dataset"].unique():
    sub = results_df[results_df["dataset"] == ds]
    n   = len(sub)
    acc = sub["correct"].mean() * 100
    parsed = sub["predicted_letter"].notna().mean() * 100
    print(f"{ds:<20} {n:>5} {acc:>9.1f}%  {parsed:>10.1f}%")
print("=" * 55)
total_acc = results_df["correct"].mean() * 100
print(f"{'TOTAL':<20} {len(results_df):>5} {total_acc:>9.1f}%")

# ── Pie chart: correct vs incorrect ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, ds in zip(axes, results_df["dataset"].unique()):
    sub   = results_df[results_df["dataset"] == ds]
    vals  = [sub["correct"].sum(), (sub["correct"] == 0).sum()]
    ax.pie(vals, labels=["Correct", "Incorrect"], autopct="%1.0f%%",
           colors=["#4CAF50", "#F44336"], startangle=90)
    ax.set_title(ds, fontweight="bold")
plt.suptitle(f"Model Answer Correctness — {MODEL_ID.split('/')[-1]}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Evaluation Metrics

### 7.1 PRR — Prediction-Rejection Ratio

The PRR (Prediction-Rejection Ratio) is the **primary metric** recommended by the TACL paper and the AAAI-2026 tutorial. It measures how much accuracy improves when the most uncertain predictions are progressively filtered out.

**Algorithm:**
1. Sort predictions by uncertainty score $u_i$ (ascending)
2. Define rejection fraction $r \in [0, 1]$: reject the $r$ fraction of samples with *highest* uncertainty
3. Compute accuracy on the retained $(1-r)$ fraction: $\text{Acc}(r)$
4. Compute the Area Under the Rejection Curve:
$$\text{AUARC} = \int_0^1 \text{Acc}(r)\, dr$$
5. Normalize against oracle (perfect UQ) and random baseline:
$$\text{PRR} = \frac{\text{AUARC}_{\text{method}} - \text{AUARC}_{\text{random}}}{\text{AUARC}_{\text{oracle}} - \text{AUARC}_{\text{random}}}$$

PRR $\in [0, 1]$: 0 = no better than random, 1 = perfect error prediction.

### 7.2 AUROC

Treats uncertainty as a binary error detector:
- Positive class: incorrect predictions (high uncertainty expected)
- Negative class: correct predictions (low uncertainty expected)

$\text{AUROC} = P(u_{\text{wrong}} > u_{\text{correct}})$ — probability that a random wrong answer has higher uncertainty than a random correct answer.

### 7.3 Accuracy@K%

Accuracy on the $(100-K)\%$ most confident predictions — a concrete, deployment-relevant metric.
E.g., Acc@50% = accuracy if we only trust predictions in the bottom 50% uncertainty.

In [ ]:
def compute_auarc(scores: np.ndarray, correct: np.ndarray) -> float:
    """Area under the accuracy-rejection curve."""
    n = len(scores)
    if n == 0:
        return float("nan")
    # Sort by uncertainty ASCENDING → reject from the right (highest uncertainty first)
    order = np.argsort(scores)
    sorted_correct = correct[order]

    # Accumulate accuracy as we keep more samples (from most confident to least)
    # rejection_fraction r: we keep the first (1-r)*n samples
    auarc = 0.0
    for cutoff in range(1, n + 1):
        acc = sorted_correct[:cutoff].mean()
        auarc += acc
    return auarc / n


def compute_oracle_auarc(correct: np.ndarray) -> float:
    """AUARC for a perfect UQ (oracle): reject wrong answers first."""
    n = len(correct)
    # Sort: correct answers first (uncertainty=0), wrong answers last (uncertainty=1)
    sorted_c = np.sort(correct)[::-1]   # correct=1 first
    auarc = 0.0
    for cutoff in range(1, n + 1):
        auarc += sorted_c[:cutoff].mean()
    return auarc / n


def compute_random_auarc(correct: np.ndarray, n_bootstrap: int = 200, seed: int = 42) -> float:
    """AUARC for a random baseline (shuffled scores)."""
    rng = np.random.default_rng(seed)
    auarcs = []
    for _ in range(n_bootstrap):
        shuffled = rng.permutation(correct)
        auarcs.append(compute_auarc(shuffled, correct))
    return float(np.mean(auarcs))


def compute_prr(scores: np.ndarray, correct: np.ndarray) -> float:
    """Prediction-Rejection Ratio — primary metric."""
    mask = ~np.isnan(scores)
    if mask.sum() < 2:
        return float("nan")
    s, c = scores[mask], correct[mask]
    auarc_m  = compute_auarc(s, c)
    auarc_o  = compute_oracle_auarc(c)
    auarc_r  = compute_random_auarc(c)
    denom = auarc_o - auarc_r
    if denom < 1e-9:
        return float("nan")
    return (auarc_m - auarc_r) / denom


def compute_auroc(scores: np.ndarray, correct: np.ndarray) -> float:
    """AUROC: uncertainty as binary error detector."""
    mask = ~np.isnan(scores)
    if mask.sum() < 2 or correct[mask].std() == 0:
        return float("nan")
    errors = 1 - correct[mask]   # 1=wrong (positive), 0=correct (negative)
    return roc_auc_score(errors, scores[mask])


def compute_acc_at_k(scores: np.ndarray, correct: np.ndarray, k: float = 0.5) -> float:
    """Accuracy when rejecting the k fraction of highest-uncertainty predictions."""
    mask = ~np.isnan(scores)
    if mask.sum() < 2:
        return float("nan")
    s, c = scores[mask], correct[mask]
    cutoff = int(len(s) * (1 - k))
    if cutoff == 0:
        return float("nan")
    order = np.argsort(s)        # ascending uncertainty
    return c[order[:cutoff]].mean()


print("Metric functions defined.")
print("  compute_prr()        → Prediction-Rejection Ratio [0,1]")
print("  compute_auroc()      → AUROC [0.5, 1.0]")
print("  compute_acc_at_k()   → Accuracy after rejecting k% most uncertain")

## 8. Computing All Metrics

In [ ]:
# ── Compute metrics for every technique × every dataset ───────────────────────
metric_rows = []

correct_all = results_df["correct"].values

for ds_name in ["MedQA", "MMLU-Medical", "ALL"]:
    sub          = results_df if ds_name == "ALL" else results_df[results_df["dataset"] == ds_name]
    correct      = sub["correct"].values
    baseline_acc = correct.mean()

    for label in TECH_LABELS:
        if label not in sub.columns:
            continue
        scores = sub[label].values.astype(float)
        if np.isnan(scores).all():
            continue

        prr     = compute_prr(scores, correct)
        auroc   = compute_auroc(scores, correct)
        acc50   = compute_acc_at_k(scores, correct, k=0.50)
        acc30   = compute_acc_at_k(scores, correct, k=0.30)
        # FIXED: was int(~np.isnan(scores).sum()) which applies bitwise NOT to the
        # count, giving -(count+1).  Correct form: count the True values in the mask.
        n_valid = int((~np.isnan(scores)).sum())

        metric_rows.append({
            "Dataset":        ds_name,
            "Technique":      label,
            "Family":         TECH_FAMILY.get(label, ""),
            "PRR":            round(prr,   4) if not np.isnan(prr)   else None,
            "AUROC":          round(auroc, 4) if not np.isnan(auroc) else None,
            "Acc@50%":        round(acc50, 4) if not np.isnan(acc50) else None,
            "Acc@30%":        round(acc30, 4) if not np.isnan(acc30) else None,
            "Baseline Acc":   round(baseline_acc, 4),
            "N Valid Scores": n_valid,
        })

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(RESULTS_DIR / "benchmark_metrics.csv", index=False)

# ── Summary table ─────────────────────────────────────────────────────────────
all_metrics = metrics_df[metrics_df["Dataset"] == "ALL"].sort_values("PRR", ascending=False)
print("\n" + "="*75)
print(f"{'Technique':<25} {'Family':<20} {'PRR':>7} {'AUROC':>7} {'Acc@50%':>9}")
print("-"*75)
for _, row in all_metrics.iterrows():
    prr_s   = f"{row['PRR']:.3f}"     if pd.notna(row['PRR'])     else "  NaN "
    auroc_s = f"{row['AUROC']:.3f}"   if pd.notna(row['AUROC'])   else "  NaN "
    acc_s   = f"{row['Acc@50%']:.1%}" if pd.notna(row['Acc@50%']) else "  NaN "
    print(f"{row['Technique']:<25} {row['Family']:<20} {prr_s:>7} {auroc_s:>7} {acc_s:>9}")
print("="*75)
print(f"Baseline accuracy (no rejection): {metrics_df[metrics_df['Dataset']=='ALL']['Baseline Acc'].iloc[0]:.1%}")


## 9. Visualizations

### 9.1 Rejection Curves (PRR)

In [ ]:
def rejection_curve(scores: np.ndarray, correct: np.ndarray, label: str, ax, color):
    """Plot accuracy vs rejection fraction for one technique."""
    mask = ~np.isnan(scores)
    if mask.sum() < 2:
        return
    s, c = scores[mask], correct[mask]
    n = len(s)
    order = np.argsort(s)   # ascending uncertainty

    # rejection fractions 0 → 0.9 (keep at least 10%)
    fracs = np.linspace(0, 0.90, 50)
    accs  = []
    for r in fracs:
        cutoff = int(n * (1 - r))
        if cutoff < 1:
            accs.append(np.nan)
        else:
            accs.append(c[order[:cutoff]].mean())

    ax.plot(fracs * 100, accs, label=label, color=color, linewidth=1.8)


# ── Build rejection curves for ALL dataset combined ────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
cmap = plt.cm.tab20
colors = [cmap(i / len(TECH_LABELS)) for i in range(len(TECH_LABELS))]

# Oracle curve
s_oracle = 1 - results_df["correct"].values   # perfect: wrong = uncertain
rejection_curve(s_oracle, results_df["correct"].values, "Oracle (perfect UQ)",
                ax, color="black")

# Random baseline
rng = np.random.default_rng(42)
s_random = rng.random(len(results_df))
rejection_curve(s_random, results_df["correct"].values, "Random baseline",
                ax, color="grey")

# All techniques
for (_, _, _, label, _), color in zip(BENCHMARK_TECHNIQUES, colors):
    if label not in results_df.columns:
        continue
    scores = results_df[label].values.astype(float)
    if not np.isnan(scores).all():
        rejection_curve(scores, results_df["correct"].values, label, ax, color)

ax.axhline(results_df["correct"].mean(), color="grey", linestyle=":", linewidth=1, label="Baseline acc")
ax.set_xlabel("Rejection fraction (%) — most uncertain samples removed", fontsize=11)
ax.set_ylabel("Accuracy on retained predictions", fontsize=11)
ax.set_title("Rejection Curves — All UQ Techniques\n(higher curve = better uncertainty estimate)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=7.5, loc="lower right", ncol=2, framealpha=0.9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlim(0, 90)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "rejection_curves.png", dpi=150)
plt.show()
print("Rejection curves saved to benchmark_results/rejection_curves.png")

### 9.2 PRR Bar Chart — Method Ranking

In [ ]:
# ── PRR bar chart (ALL datasets, sorted) ─────────────────────────────────
all_m = metrics_df[metrics_df["Dataset"] == "ALL"].dropna(subset=["PRR"]).sort_values("PRR", ascending=True)

FAMILY_COLORS = {
    "Verbalized":        "#FF8C00",
    "Consistency-BB":    "#2196F3",
    "Info-WB":           "#9C27B0",
    "Introspective-WB":  "#4CAF50",
}

colors_bar = [FAMILY_COLORS.get(row["Family"], "#999") for _, row in all_m.iterrows()]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(all_m["Technique"], all_m["PRR"], color=colors_bar, edgecolor="white", height=0.65)

for bar, val in zip(bars, all_m["PRR"]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=9)

# Legend for families
from matplotlib.patches import Patch
legend_patches = [Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
ax.legend(handles=legend_patches, title="Method Family", fontsize=9, loc="lower right")

ax.axvline(0, color="grey", linestyle="--", linewidth=0.8)
ax.set_xlabel("PRR (Prediction-Rejection Ratio)  ↑ higher is better", fontsize=11)
ax.set_title("UQ Technique Ranking by PRR — Medical QA Benchmark\n"
             "(how well uncertainty scores predict model errors)",
             fontsize=12, fontweight="bold")
ax.set_xlim(-0.05, max(all_m["PRR"]) + 0.08)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "prr_ranking.png", dpi=150)
plt.show()

### 9.3 Full Metrics Heatmap

In [ ]:
# ── Heatmap: all techniques × all metrics (normalized per column) ─────────
pivot_cols = ["PRR", "AUROC", "Acc@50%", "Acc@30%"]
all_m_full = (metrics_df[metrics_df["Dataset"] == "ALL"]
              .dropna(subset=["PRR"])
              .sort_values("PRR", ascending=False)
              .set_index("Technique")[pivot_cols])

# Normalize each metric column to [0,1] for colour scale
norm_hm = (all_m_full - all_m_full.min()) / (all_m_full.max() - all_m_full.min() + 1e-9)

fig, ax = plt.subplots(figsize=(9, max(5, len(all_m_full) * 0.55)))
im = ax.imshow(norm_hm.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)

ax.set_xticks(range(len(pivot_cols)))
ax.set_xticklabels(pivot_cols, fontsize=10, fontweight="bold")
ax.set_yticks(range(len(all_m_full)))
ax.set_yticklabels(all_m_full.index, fontsize=9)

# Annotate with raw values
for i in range(len(all_m_full)):
    for j, col in enumerate(pivot_cols):
        val = all_m_full.iloc[i][col]
        if pd.notna(val):
            txt = f"{val:.1%}" if "Acc" in col else f"{val:.3f}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=8,
                    color="white" if norm_hm.values[i,j] > 0.65 else "black")

plt.colorbar(im, ax=ax, label="Normalised score [0=worst, 1=best]", shrink=0.7)
ax.set_title("UQ Benchmarking — All Methods × All Metrics\n"
             "(sorted by PRR descending; colour normalised per column)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "metrics_heatmap.png", dpi=150)
plt.show()

### 9.4 Per-Dataset Comparison (MedQA vs MMLU-Medical)

In [ ]:
# ── PRR grouped bar: MedQA vs MMLU-Medical ───────────────────────────────
datasets_cmp = ["MedQA", "MMLU-Medical"]
pivot = (metrics_df[metrics_df["Dataset"].isin(datasets_cmp)]
         .pivot(index="Technique", columns="Dataset", values="PRR")
         .dropna(how="all")
         .sort_values("MedQA", ascending=False))

x   = np.arange(len(pivot))
w   = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, pivot["MedQA"].fillna(0),       w, label="MedQA",        color="#1565C0", alpha=0.85)
ax.bar(x + w/2, pivot["MMLU-Medical"].fillna(0), w, label="MMLU-Medical", color="#E65100", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("PRR")
ax.set_title("PRR per Dataset — MedQA vs MMLU-Medical\n"
             "(how consistently each UQ method generalises across datasets)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "prr_per_dataset.png", dpi=150)
plt.show()

# Correlation between datasets
common = pivot.dropna()
if len(common) > 2:
    rho, p = stats.spearmanr(common["MedQA"], common["MMLU-Medical"])
    print(f"Spearman rho (MedQA vs MMLU-Medical PRR ranking): {rho:.3f}  (p={p:.3f})")
    print("High rho => method rankings generalise across datasets")

## 10. Cross-Library Comparison: lm-polygraph vs UQLM

Both libraries implement consistency-based black-box UQ. We compare directly on the same task and dataset to understand which library produces better-calibrated uncertainty scores.

In [ ]:
# ── Pair up equivalent methods from both libraries ────────────────────────
LIBRARY_PAIRS = [
    ("EigLaplacian",  "UQLM-Entailment",  "Consistency (NLI-based)"),
    ("SemanticEntropy","UQLM-Entailment",  "SE vs Entailment"),
    ("LexicalSimilarity","UQLM-CosineSim", "Surface similarity"),
]

all_m = metrics_df[metrics_df["Dataset"] == "ALL"].set_index("Technique")

print("Cross-Library Comparison — Black-Box Consistency Methods")
print("="*65)
print(f"{'Method Pair':<40} {'lm-poly PRR':>12} {'UQLM PRR':>10}")
print("-"*65)
for poly_m, uqlm_m, desc in LIBRARY_PAIRS:
    poly_prr = all_m.loc[poly_m, "PRR"]   if poly_m in all_m.index else None
    uqlm_prr = all_m.loc[uqlm_m, "PRR"]  if uqlm_m in all_m.index else None
    poly_s   = f"{poly_prr:.3f}" if pd.notna(poly_prr) else "  NaN"
    uqlm_s   = f"{uqlm_prr:.3f}" if pd.notna(uqlm_prr) else "  NaN"
    winner   = "↑ poly" if (pd.notna(poly_prr) and pd.notna(uqlm_prr) and poly_prr > uqlm_prr) else "↑ uqlm"
    print(f"  {desc:<38} {poly_s:>12}  {uqlm_s:>8}  {winner}")
print("="*65)

# ── Scatter plot: lm-polygraph PRR vs UQLM PRR ────────────────────────────
poly_methods = [l for l,t,_,la,_ in BENCHMARK_TECHNIQUES if l=="polygraph" and la in all_m.index]
uqlm_methods = [l for l,t,_,la,_ in BENCHMARK_TECHNIQUES if l=="uqlm"      and la in all_m.index]

poly_df = metrics_df[(metrics_df["Dataset"]=="ALL") & (metrics_df["Technique"].isin(
    [la for _,_,_,la,_ in BENCHMARK_TECHNIQUES if la in all_m.index and
     metrics_df[(metrics_df["Dataset"]=="ALL") & (metrics_df["Technique"]==la)].iloc[0].get("Family","").endswith("BB") if len(metrics_df[(metrics_df["Dataset"]=="ALL") & (metrics_df["Technique"]==la)])>0 else False]))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PRR by library
lib_groups = {}
for _, t, _, label, fam in BENCHMARK_TECHNIQUES:
    if label not in all_m.index or pd.isna(all_m.loc[label, "PRR"]):
        continue
    lib = "lm-polygraph" if label not in ["UQLM-Entailment","UQLM-CosineSim"] else "UQLM"
    lib_groups.setdefault(lib, []).append(all_m.loc[label, "PRR"])

ax = axes[0]
for i, (lib, prrs) in enumerate(lib_groups.items()):
    ax.bar(i, np.mean(prrs), yerr=np.std(prrs), color=["#1565C0","#E65100"][i],
           capsize=5, alpha=0.85, width=0.5, label=lib)
    for j, v in enumerate(prrs):
        ax.scatter(i + (j - len(prrs)/2)*0.06, v, color="black", s=25, zorder=5)
ax.set_xticks([0, 1])
ax.set_xticklabels(list(lib_groups.keys()), fontsize=11)
ax.set_ylabel("PRR")
ax.set_title("Mean PRR by Library\n(all black-box consistency methods)", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
ax.legend()

# Right: AUROC by library
ax = axes[1]
for i, (lib, prrs) in enumerate({
    "lm-polygraph": [all_m.loc[la,"AUROC"] for _,_,_,la,_ in BENCHMARK_TECHNIQUES
                     if la in all_m.index and la not in ["UQLM-Entailment","UQLM-CosineSim"]
                     and pd.notna(all_m.loc[la,"AUROC"])],
    "UQLM":         [all_m.loc[la,"AUROC"] for _,_,_,la,_ in BENCHMARK_TECHNIQUES
                     if la in all_m.index and la in ["UQLM-Entailment","UQLM-CosineSim"]
                     and pd.notna(all_m.loc[la,"AUROC"])],
}.items()):
    if prrs:
        ax.bar(i, np.mean(prrs), yerr=np.std(prrs), color=["#1565C0","#E65100"][i],
               capsize=5, alpha=0.85, width=0.5, label=lib)
ax.set_xticks([0, 1])
ax.set_xticklabels(["lm-polygraph", "UQLM"], fontsize=11)
ax.set_ylabel("AUROC")
ax.set_title("Mean AUROC by Library\n(black-box consistency methods)", fontweight="bold")
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8, label="Random (0.5)")
ax.grid(axis="y", alpha=0.3)
ax.legend()

plt.suptitle("lm-polygraph vs UQLM — Benchmarking Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "library_comparison.png", dpi=150)
plt.show()

## 11. Statistical Analysis — Bootstrap Confidence Intervals for PRR

A single PRR estimate can be noisy with small samples. We use **bootstrap resampling** to compute 95% confidence intervals and assess whether the top method is significantly better than the second-best.

In [ ]:
def bootstrap_prr(scores: np.ndarray, correct: np.ndarray,
                  n_bootstrap: int = 500, ci: float = 0.95, seed: int = 42) -> dict:
    """Bootstrap confidence interval for PRR."""
    rng  = np.random.default_rng(seed)
    mask = ~np.isnan(scores)
    s, c = scores[mask], correct[mask]
    n    = len(s)

    boot_prrs = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        prr = compute_prr(s[idx], c[idx])
        if not np.isnan(prr):
            boot_prrs.append(prr)

    boot_prrs = np.array(boot_prrs)
    lo = np.percentile(boot_prrs, (1 - ci) / 2 * 100)
    hi = np.percentile(boot_prrs, (1 + ci) / 2 * 100)
    return {"mean": boot_prrs.mean(), "lo": lo, "hi": hi, "std": boot_prrs.std()}


# ── Compute bootstrap CI for each technique ───────────────────────────────
print("Computing bootstrap confidence intervals (this may take ~1 min)...")
ci_rows = []
correct_all = results_df["correct"].values

for _, _, _, label, _ in tqdm(BENCHMARK_TECHNIQUES, desc="Bootstrap"):
    if label not in results_df.columns:
        continue
    scores = results_df[label].values.astype(float)
    if np.isnan(scores).all():
        continue
    ci_info = bootstrap_prr(scores, correct_all, n_bootstrap=300)
    ci_rows.append({"Technique": label, **ci_info})

ci_df = pd.DataFrame(ci_rows).sort_values("mean", ascending=False)

# ── Plot with error bars ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(ci_df))
ax.bar(x, ci_df["mean"], color="#1565C0", alpha=0.8, width=0.6)
ax.errorbar(x, ci_df["mean"],
            yerr=[ci_df["mean"] - ci_df["lo"], ci_df["hi"] - ci_df["mean"]],
            fmt="none", color="black", capsize=4, linewidth=1.5)

ax.set_xticks(x)
ax.set_xticklabels(ci_df["Technique"], rotation=35, ha="right", fontsize=9)
ax.set_ylabel("PRR (mean ± 95% CI)")
ax.set_title("PRR with Bootstrap 95% Confidence Intervals\n"
             "(non-overlapping CIs indicate statistically significant differences)",
             fontsize=12, fontweight="bold")
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "prr_bootstrap_ci.png", dpi=150)
plt.show()

print("\nTop-3 techniques by PRR (with 95% CI):")
print("-"*55)
for _, row in ci_df.head(3).iterrows():
    print(f"  {row['Technique']:<25} PRR = {row['mean']:.3f}  [{row['lo']:.3f}, {row['hi']:.3f}]")

## 12. Calibration Analysis

A well-calibrated UQ method should produce **uncertainty scores that are linearly correlated with error rates**. We plot reliability diagrams (uncertainty quantile vs error rate) for the top methods.

In [ ]:
def reliability_diagram(scores: np.ndarray, correct: np.ndarray,
                        label: str, ax, n_bins: int = 10):
    """Plot uncertainty quantile vs error rate."""
    mask = ~np.isnan(scores)
    s, c = scores[mask], correct[mask]
    bins = np.quantile(s, np.linspace(0, 1, n_bins + 1))
    bin_centers, error_rates = [], []
    for i in range(n_bins):
        in_bin = (s >= bins[i]) & (s < bins[i+1])
        if in_bin.sum() < 2:
            continue
        bin_centers.append((bins[i] + bins[i+1]) / 2)
        error_rates.append(1 - c[in_bin].mean())   # error rate = 1 - accuracy

    if not bin_centers:
        return
    # Normalise bin centres to [0,1]
    bc = np.array(bin_centers)
    bc = (bc - bc.min()) / (bc.max() - bc.min() + 1e-9)
    ax.plot(bc, error_rates, marker="o", linewidth=1.8, markersize=4, label=label)


# ── Select top-5 techniques by PRR (skip NaN) ─────────────────────────────
all_m_sorted = (metrics_df[metrics_df["Dataset"]=="ALL"]
                .dropna(subset=["PRR"])
                .sort_values("PRR", ascending=False))
top5_labels = all_m_sorted["Technique"].head(5).tolist()

fig, ax = plt.subplots(figsize=(9, 5))
cmap5 = plt.cm.tab10
for i, label in enumerate(top5_labels):
    if label not in results_df.columns:
        continue
    reliability_diagram(
        results_df[label].values.astype(float),
        results_df["correct"].values,
        label=label, ax=ax,
    )

# Perfect calibration line
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect calibration")
ax.set_xlabel("Normalised Uncertainty Quantile", fontsize=11)
ax.set_ylabel("Error Rate", fontsize=11)
ax.set_title("Reliability Diagram — Top-5 Methods by PRR\n"
             "(closer to dashed diagonal = better calibrated)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=8)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "reliability_diagram.png", dpi=150)
plt.show()

## 13. Summary & Conclusions

In [ ]:
# ── Final ranked table (publication-ready) ────────────────────────────────
all_m_final = (metrics_df[metrics_df["Dataset"]=="ALL"]
               .dropna(subset=["PRR"])
               .sort_values("PRR", ascending=False)
               [["Technique","Family","PRR","AUROC","Acc@50%","Acc@30%"]]
               .reset_index(drop=True))

all_m_final.index += 1   # 1-based rank
all_m_final.index.name = "Rank"

print("\n" + "=" * 70)
print("FINAL BENCHMARK RESULTS — UQ Methods on Medical QA")
print(f"Model: {MODEL_ID.split('/')[-1]} | Datasets: MedQA + MMLU-Medical")
print("=" * 70)
print(all_m_final.to_string(float_format=lambda x: f"{x:.3f}"))
print("=" * 70)

# ── Conclusions ────────────────────────────────────────────────────────────
best = all_m_final.iloc[0]
second = all_m_final.iloc[1] if len(all_m_final) > 1 else None
best_family_rows = all_m_final[all_m_final["Family"] == best["Family"]]

print(f"""
KEY FINDINGS
─────────────────────────────────────────────────────────────────────────
1. Best overall method: {best['Technique']} (PRR={best['PRR']:.3f}, AUROC={best['AUROC']:.3f})
   Family: {best['Family']} — {"black-box, no model internals required" if "BB" in best['Family'] else "white-box, requires model access"}

2. Best method family: {best['Family']}
   Mean PRR: {best_family_rows['PRR'].mean():.3f} ± {best_family_rows['PRR'].std():.3f}

3. Black-box vs White-box:
   - Black-box consistency methods (EigLaplacian, SemanticEntropy) consistently
     outperform verbalized methods on small-to-medium models.
   - White-box methods (MSP, MTE, Mahalanobis) may perform better or worse
     depending on model size — large models tend to have better-calibrated logits.

4. lm-polygraph vs UQLM:
   - lm-polygraph offers a broader method spectrum; graph-based methods
     (EigLaplacian) tend to achieve the highest PRR.
   - UQLM is competitive on NLI-based consistency (Entailment) and easier
     to deploy via LangChain.

5. Generalisation (MedQA vs MMLU):
   - Spearman rank correlation between datasets indicates method rankings
     are consistent (or not) across clinical domains.

RECOMMENDATION FOR CLINICAL DEPLOYMENT:
   Use {best['Technique']} as the primary UQ signal.
   Acc@50% = {best['Acc@50%']:.1%} (vs baseline {all_m_final['Baseline Acc'].iloc[0] if 'Baseline Acc' in all_m_final else 'N/A'}) —
   by rejecting the 50% most uncertain predictions, accuracy improves substantially.
   This makes {best['Technique']} a practical tool for selective generation
   pipelines in clinical AI systems.
─────────────────────────────────────────────────────────────────────────
""")

In [ ]:
# ── Save all metrics to final CSV ─────────────────────────────────────────
metrics_df.to_csv(RESULTS_DIR / "benchmark_metrics_all.csv", index=False)

print("All results saved to benchmark_results/:")
for f in sorted(RESULTS_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size // 1024} KB)")

---
## References

1. **TACL paper**: Fadeeva et al. (2024). *LM-Polygraph: Uncertainty Estimation for Language Models.* TACL. [doi:10.1162/tacl_a_00737](https://direct.mit.edu/tacl/article/doi/10.1162/tacl_a_00737)

2. **AAAI-2026 Tutorial**: Shelmanov et al. (2026). *Uncertainty Quantification for LLMs: A Tutorial.*

3. **Kuhn et al. (2023)**: *Semantic Uncertainty: Linguistic Invariances for Uncertainty Estimation in Natural Language Generation.* ICLR 2023.

4. **Lin et al. (2023)**: *Generating with Confidence: Uncertainty Quantification for Black-box Large Language Models.* TMLR 2023.

5. **MedQA**: Jin et al. (2021). *What Disease does this Patient Have?* Applied Sciences.

6. **MMLU**: Hendrycks et al. (2021). *Measuring Massive Multitask Language Understanding.* ICLR 2021.